# M2: Entrenamiento de Segmentación de Vehículos (YOLO11s-seg)
Clases: `moto`, `auto`, `minibus`, `camion`, `bus`.


In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/peaje/evidencias /content/drive/MyDrive/peaje/modelos /content/drive/MyDrive/peaje/runs
!git clone https://github.com/Adiel05/Control_Peaje.git /content/Control_Peaje 2>/dev/null || git -C /content/Control_Peaje pull
%cd /content/Control_Peaje
!pip -q install -r entrenamiento/requirements.txt
!python scripts/diagnostico.py > /content/drive/MyDrive/peaje/evidencias/diagnostico_entrenamiento.txt
!cat /content/drive/MyDrive/peaje/evidencias/diagnostico_entrenamiento.txt


In [ ]:
from google.colab import userdata
from roboflow import Roboflow

# Configura estos datos con tu proyecto de Roboflow
WORKSPACE = 'tu-workspace'
PROYECTO = 'peaje-vehiculos-seg'
VERSION = 1

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11')


In [ ]:
import yaml
yaml_path = f'{ds.location}/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)
print('Clases encontradas en data.yaml:', data_cfg.get('names'))
esperadas = {'moto', 'auto', 'minibus', 'camion', 'bus'}
nombres = set(data_cfg['names'].values()) if isinstance(data_cfg['names'], dict) else set(data_cfg['names'])
assert esperadas.issubset({n.lower() for n in nombres}), f'Faltan clases: {esperadas - nombres}'


In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11s-seg.pt')
modelo.train(
    data=yaml_path,
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    project='/content/drive/MyDrive/peaje/runs',
    name='m2_seg',
)


In [ ]:
import json, shutil
best_pt = '/content/drive/MyDrive/peaje/runs/m2_seg/weights/best.pt'
mejor = YOLO(best_pt)
metricas = mejor.val(data=yaml_path)
resumen = {
    'box_map50': float(metricas.box.map50),
    'seg_map50': float(metricas.seg.map50),
    'seg_map': float(metricas.seg.map),
    'clases': metricas.names,
    'seg_map50_por_clase': [float(v) for v in metricas.seg.ap50],
}
print(json.dumps(resumen, indent=2, ensure_ascii=False))
with open('/content/drive/MyDrive/peaje/evidencias/metricas_m2.json', 'w') as f:
    json.dump(resumen, f, indent=2)
shutil.copy(best_pt, '/content/drive/MyDrive/peaje/modelos/m2_seg.pt')
print('Modelo guardado en Drive: m2_seg.pt')


In [ ]:
import glob
from IPython.display import Image, display
imgs_val = glob.glob(f'{ds.location}/valid/images/*')[:4]
preds = mejor.predict(imgs_val, save=True, project='/tmp/m2_vis', name='val')
for p in glob.glob('/tmp/m2_vis/val/*.jpg')[:4]:
    display(Image(filename=p))
